In [1]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from brainrender._io import load_mesh_from_file
from myterial import orange
from rich import print
from brainglobe_atlasapi import BrainGlobeAtlas
from brainrender import Scene
import brainrender
from brainrender.actors import Points, Line
import numpy as np
from utils import slice_util
from PIL import Image
import os
import subprocess
import  vedo 
import pandas as pd
import brainglobe_heatmap as bgh
import matplotlib.pyplot as plt
import matplotlib
from tqdm.notebook import trange, tqdm
from chamferdist import ChamferDistance
import torch
import multiprocessing


In [2]:
bg_atlas = BrainGlobeAtlas("allen_mouse_50um", check_latest=False)

In [5]:
len(bg_atlas.structures)

840

In [4]:
df_id_name = pd.read_csv('resources/id_name.csv')
id_2_name = {}
name_2_id = {}
for _i, _row in df_id_name[['Region ID','Region name']].iterrows():
    id_2_name[_row['Region ID']] = _row['Region name']
    name_2_id[_row['Region name']] = _row['Region ID']
id_2_name


{
    0: 'Clear Label',
    997: 'root',
    8: 'Basic cell groups and regions',
    567: 'Cerebrum',
    688: 'Cerebral cortex',
    695: 'Cortical plate',
    315: 'Isocortex',
    184: 'Frontal pole, cerebral cortex',
    68: 'Frontal pole, layer 1',
    667: 'Frontal pole, layer 2/3',
    526157192: 'Frontal pole, layer 5',
    526157196: 'Frontal pole, layer 6a',
    526322264: 'Frontal pole, layer 6b',
    500: 'Somatomotor areas',
    107: 'Somatomotor areas, Layer 1',
    219: 'Somatomotor areas, Layer 2/3',
    299: 'Somatomotor areas, Layer 5',
    644: 'Somatomotor areas, Layer 6a',
    947: 'Somatomotor areas, Layer 6b',
    985: 'Primary motor area',
    320: 'Primary motor area, Layer 1',
    943: 'Primary motor area, Layer 2/3',
    648: 'Primary motor area, Layer 5',
    844: 'Primary motor area, Layer 6a',
    882: 'Primary motor area, Layer 6b',
    993: 'Secondary motor area',
    656: 'Secondary motor area, layer 1',
    962: 'Secondary motor area, layer 2/3',
    

In [11]:
df_total = pd.read_csv('output/df_total.csv')
df_total

,color,veh_exp,0,997,8,567,688,695,315,184,...,11,18,25,34,43,49,57,65,624,304325711
0,PV,0,0.0,0.004465,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,PV,1,0.0,0.003177,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,PV-cfos,0,0.0,0.000439,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,PV-cfos,1,0.0,0.000579,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,PV/cfos fraction,0,NaN,9.762396,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PV/cfos fraction,1,0.0,16.432816,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,SST,0,0.0,0.008105,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,SST,1,0.0,0.008052,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,SST-PV,0,0.0,0.002011,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,SST-PV,1,0.0,0.000679,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
bg_atlas.structures[480149202]

KeyError: 480149202

In [5]:
region_id_matched_atlas_list = []
id_list = []
name_list = []
num_of_pts_list = []
for _index, _region_name in enumerate(tqdm(bg_atlas.structures)):
    if len(bg_atlas.get_structure_descendants(_region_name))==0:
        try:
            num_of_pts_list.append(len(bg_atlas.structures[_region_name]['mesh'].points))
            id_list.append(bg_atlas.structures[_region_name]['id'])

            _region_full_name = bg_atlas.structures[_region_name]['name']

            name_list.append(_region_full_name)
            
            region_id_matched_atlas_list.append(name_2_id[_region_full_name])
        except:
            pass
region_id_matched_atlas_list = [str(a) for a in  region_id_matched_atlas_list]

with open('resources/region_id_matched_atlas_list.txt', 'w') as f:
    for line in region_id_matched_atlas_list:
        f.write(f"{line}\n")

print('the number of leaf in Atlas:',len(id_list))
print('the number of leaf in both Atlas and tg:',len(region_id_matched_atlas_list))

  0%|          | 0/840 [00:00<?, ?it/s]

the number of leaf in Atlas: 642

the number of leaf in both Atlas and tg: 642

In [13]:
df_total = df_total.loc[:, ['color','veh_exp']+region_id_matched_atlas_list]

In [45]:
color = 'PV'
veh_exp = 0
value_list = df_total.query(f"color=='{color}' and veh_exp=={0}")[df_total.columns[2:]].values[0]
np.save("values_veh.npy",value_list)
value_list = df_total.query(f"color=='{color}' and veh_exp=={1}")[df_total.columns[2:]].values[0]
np.save("values_exp.npy",value_list)

In [37]:
np.load("values.npy", mmap_mode='r')

#with open("values.npy", mmap_mode='r') as dt:
#    data = np.load(dt)


memmap([5.82500e-04, 1.00232e-02, 1.92836e-02, 1.76761e-02, 7.19100e-04,
        1.07520e-03, 2.76469e-02, 3.44791e-02, 2.38208e-02, 2.78920e-03,
        1.32710e-03, 2.09847e-02, 3.02227e-02, 2.54964e-02, 4.17770e-03,
        1.21990e-03, 1.77589e-02, 3.27503e-02, 3.86159e-02, 2.48112e-02,
        2.68270e-03, 1.02700e-03, 1.83254e-02, 3.34552e-02, 3.93111e-02,
        2.60468e-02, 4.95990e-03, 1.13210e-03, 2.79090e-02, 4.35707e-02,
        3.99879e-02, 3.11896e-02, 3.75890e-03, 1.15330e-03, 1.96066e-02,
        3.53221e-02, 3.75358e-02, 2.37308e-02, 4.79310e-03, 9.17200e-04,
        2.47057e-02, 4.14796e-02, 3.83576e-02, 2.37583e-02, 2.13960e-03,
        1.60140e-03, 2.76270e-02, 4.12796e-02, 4.12949e-02, 3.23348e-02,
        7.23550e-03, 1.29110e-03, 2.48644e-02, 3.67270e-02, 3.97225e-02,
        2.57704e-02, 1.71310e-03, 1.11950e-03, 1.78813e-02, 3.59236e-02,
        3.34595e-02, 2.06431e-02, 2.77620e-03, 6.63700e-04, 1.33132e-02,
        2.52678e-02, 1.39908e-02, 7.58650e-03, 4.5

In [25]:
chamferDist = ChamferDistance()


id_list =  list(df_total.iloc[0].index[2:])

def process_task(_index_src, _region_id_src, _region_id_tar_list):
    global bg_atlas

    source_cloud = bg_atlas.structures[_region_id_src]['mesh'].points

    source_cloud = torch.from_numpy(np.array(source_cloud, dtype=np.float32)).unsqueeze(dim=0)
    dist_list = []
    #print(_region_id_tar_list)
    for _index_tar, _region_id_tar in enumerate(_region_id_tar_list):
   
        if _region_id_src == _region_id_tar:
            dist_list.append(0.0)
            continue
        target_cloud = bg_atlas.structures[_region_id_tar]['mesh'].points    


        target_cloud = torch.from_numpy(np.array(target_cloud, dtype=np.float32)).unsqueeze(dim=0)
        dist_forward = chamferDist(source_cloud,target_cloud)    
        dist_forward = dist_forward.detach().cpu().item()    
        dist_list.append(dist_forward)

    return (_index_src, dist_list)

_index_src_list = []
_region_id_src_list = []
_region_id_tar_list_list = []
for _index_src, _region_id_src in enumerate(id_list):
    _region_id_tar_list = []
    for _index_tar, _region_id_tar in enumerate(id_list):
        #if _region_id_src == _region_id_tar:
        #    continue
        _region_id_tar_list.append(_region_id_tar)
    _index_src_list.append(_index_src)
    _region_id_src_list.append(_region_id_src)
    _region_id_tar_list_list.append(_region_id_tar_list)
dist_mat = np.zeros((len(id_list),len(id_list)))

print(f'the number of jobs:{len(_index_src_list)}')
with multiprocessing.Pool() as pool: # Use a pool of 4 processes
    output = pool.starmap(process_task, zip(_index_src_list, _region_id_src_list, _region_id_tar_list_list))
    for _index_src, _dist_list in output:
        for _index_tar,  dist in enumerate(_dist_list) :
            dist_mat[_index_src,_index_tar] = dist
            dist_mat[_index_tar,_index_src] = dist
np.save("dist.npy",dist_mat)

the number of jobs:642

In [26]:
_d = np.load("dist.npy")
_d


array([[0.00000000e+00, 6.45076150e+06, 1.02268672e+08, ...,
        3.26892360e+10, 1.81654209e+11, 3.53878426e+09],
       [6.45076150e+06, 0.00000000e+00, 4.66236520e+07, ...,
        3.15637965e+10, 1.75297200e+11, 3.45847578e+09],
       [1.02268672e+08, 4.66236520e+07, 0.00000000e+00, ...,
        2.77268705e+10, 1.58382031e+11, 3.15581645e+09],
       ...,
       [3.26892360e+10, 3.15637965e+10, 2.77268705e+10, ...,
        0.00000000e+00, 1.56554127e+10, 2.67190816e+08],
       [1.81654209e+11, 1.75297200e+11, 1.58382031e+11, ...,
        1.56554127e+10, 0.00000000e+00, 8.08950160e+07],
       [3.53878426e+09, 3.45847578e+09, 3.15581645e+09, ...,
        2.67190816e+08, 8.08950160e+07, 0.00000000e+00]])

In [3]:
bg_atlas.structures['FRP2/3']

#bg_atlas.get_structure_descendants('FRP1')


{
    'acronym': 'FRP2/3',
    'id': 667,
    'name': 'Frontal pole, layer 2/3',
    'structure_id_path': [997, 8, 567, 688, 695, 315, 184, 667],
    'rgb_triplet': [38, 143, 69],
    'mesh_filename': PosixPath('/home/jhahn/.brainglobe/allen_mouse_50um_v1.2/meshes/667.obj'),
    'mesh': <meshio mesh object>
  Number of points: 435
  Number of cells:
    triangle: 862
  Point data: obj:vn
  Cell data: obj:group_ids
}